# UFC Fight Analytics & Prediction (1993–2026)

This project analyzes UFC fight data and fighter profiles to uncover performance patterns and predict fight outcomes.

Objectives:
- Analyze fight statistics and trends
- Study striking, grappling, and control dominance
- Explore fighter physical advantages
- Identify winning patterns
- Build a machine learning model to predict fight outcomes

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

plt.style.use("ggplot")

In [ ]:
fights = pd.read_csv("/kaggle/input/datasets/jossilva3110/ufc-dataset-1994-2026/ufc_gold_dataset_final.csv")
fighters = pd.read_csv("/kaggle/input/datasets/jossilva3110/ufc-dataset-1994-2026/ufc_fighters_final.csv")

fights["Event_Date"] = pd.to_datetime(fights["Event_Date"])
fighters["DOB"] = pd.to_datetime(fighters["DOB"])

fights.head()

In [ ]:
print("Fights:", fights.shape)
print("Fighters:", fighters.shape)

fights.info()
fighters.info()

In [ ]:
fights["Method"].value_counts().plot(kind="bar")

plt.title("Fight Outcomes by Method")

plt.show()

In [ ]:
methods = fights["Method"].str.split("-").str[0]

methods.value_counts().plot(kind="bar")

plt.title("KO / Submission / Decision Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(fights["Total_Fight_Time_Sec"], bins=50)

plt.title("Fight Duration Distribution")

plt.show()

In [ ]:
fights["strike_diff"] = fights["F1_Sig_Landed"] - fights["F2_Sig_Landed"]

plt.figure(figsize=(8,5))

sns.boxplot(
    x=fights["Winner"] == fights["Fighter_1"],
    y=fights["strike_diff"]
)

plt.title("Striking Difference vs Winner")

plt.show()

In [ ]:
fights["td_diff"] = fights["F1_TD_Landed"] - fights["F2_TD_Landed"]

sns.boxplot(
    x=fights["Winner"] == fights["Fighter_1"],
    y=fights["td_diff"]
)

plt.title("Takedown Advantage vs Winner")

plt.show()

In [ ]:
fights["ctrl_diff"] = fights["F1_Ctrl_Sec"] - fights["F2_Ctrl_Sec"]

sns.boxplot(
    x=fights["Winner"] == fights["Fighter_1"],
    y=fights["ctrl_diff"]
)

plt.title("Control Time Advantage")

plt.show()

In [ ]:

fighters.columns = fighters.columns.str.strip()
fights.columns = fights.columns.str.strip()


fighters_small = fighters[[
    "Fighter_Name",
    "Height",
    "Reach",
    "Weight",
    "SLpM",
    "Str_Acc",
    "TD_Avg"
]].copy()

fights = fights.merge(
    fighters_small,
    left_on="Fighter_1",
    right_on="Fighter_Name",
    how="left"
)


fights.rename(columns={
    "Height": "F1_Height",
    "Reach": "F1_Reach",
    "Weight": "F1_Weight",
    "SLpM": "F1_SLpM"
}, inplace=True)

In [ ]:
plt.figure(figsize=(8,5))

sns.scatterplot(
    x=fights["F1_Reach"],
    y=fights["F1_Sig_Landed"]
)

plt.title("Reach vs Striking Output")

plt.show()

In [ ]:

fights["F1_dominance"] = (
    fights["F1_Sig_Landed"] * 0.4 +
    fights["F1_TD_Landed"] * 0.3 +
    fights["F1_Sub_Att"] * 0.2 +
    fights["F1_Ctrl_Sec"] * 0.001
)

fights["F2_dominance"] = (
    fights["F2_Sig_Landed"] * 0.4 +
    fights["F2_TD_Landed"] * 0.3 +
    fights["F2_Sub_Att"] * 0.2 +
    fights["F2_Ctrl_Sec"] * 0.001
)

fights["dominance_diff"] = fights["F1_dominance"] - fights["F2_dominance"]

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    x=(fights["Winner"] == fights["Fighter_1"]),
    y=fights["dominance_diff"]
)

plt.title("Dominance Score vs Winner")
plt.xlabel("Did Fighter 1 Win?")
plt.ylabel("Dominance Difference")

plt.show()

In [ ]:
fights.sort_values("dominance_diff", ascending=False)[
    ["Fighter_1","Fighter_2","dominance_diff"]
].head(10)

In [ ]:
fights["close_fight"] = fights["dominance_diff"].abs()

fights.sort_values("close_fight")[
    ["Fighter_1","Fighter_2","Winner","dominance_diff"]
].head(10)

In [ ]:
df_ml = fights.copy()

df_ml = df_ml.dropna()

df_ml["target"] = (df_ml["Winner"] == df_ml["Fighter_1"]).astype(int)

features = [
"F1_Sig_Landed","F2_Sig_Landed",
"F1_TD_Landed","F2_TD_Landed",
"F1_Ctrl_Sec","F2_Ctrl_Sec",
"Total_Fight_Time_Sec"
]

X = df_ml[features]
y = df_ml["target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestClassifier()

model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, pred))

print(classification_report(y_test, pred))

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=features
)

importance.sort_values().plot(kind="barh")

plt.title("Fight Outcome Drivers")

plt.show()